# Get to Know a Dataset: ImageSafari Crop Image Collection

This notebook provides a guided introduction to the ImageSafari Crop Image Collection, a curated set of 6,081,555 field images representing 18 crops. It shows how the collection is organized, how to browse it anonymously on Amazon S3, how to load an image, and how to examine the crop distribution.


## How is the dataset organized?

Images are grouped by crop at the top level of the bucket. Deeper object-key components retain available metadata such as year, country, crop, season, capture method, capture date, and contributing centre. The precise depth can vary across sources, so applications should parse keys defensively.

Representative structure:

`s3://<S3-BUCKET-NAME>/<crop>/.../<year>:<country>:<crop>:<season>/.../<capture-method>/<capture-date>/<file>.<extension>`


## Libraries and public S3 connection

This tutorial uses `boto3`, `Pillow`, and `matplotlib`. Enter the bucket name and Region supplied by AWS before running the S3 cells.


In [ ]:
from io import BytesIO

import boto3
import matplotlib.pyplot as plt
from botocore import UNSIGNED
from botocore.config import Config
from PIL import Image

BUCKET = "<S3-BUCKET-NAME>"
REGION = "<AWS-REGION>"

s3 = boto3.client(
    "s3",
    region_name=REGION,
    config=Config(signature_version=UNSIGNED),
)


## Browse the crop prefixes

The following request lists the top-level crop prefixes without requiring AWS credentials.


In [ ]:
response = s3.list_objects_v2(Bucket=BUCKET, Delimiter="/")
crop_prefixes = [item["Prefix"] for item in response.get("CommonPrefixes", [])]
crop_prefixes


## What formats are present?

The collection contains JPEG, PNG, and WebP still images. Pillow can decode all three formats. File extensions should be handled case-insensitively, and production pipelines should catch decode errors even though the release has undergone corruption screening.


## Download and display one image

This example finds the first supported image under the `potato/` prefix, downloads it into memory, and displays it.


In [ ]:
extensions = (".jpg", ".jpeg", ".png", ".webp")
page = s3.list_objects_v2(Bucket=BUCKET, Prefix="potato/", MaxKeys=1000)
image_key = next(
    item["Key"]
    for item in page.get("Contents", [])
    if item["Key"].lower().endswith(extensions)
)

body = s3.get_object(Bucket=BUCKET, Key=image_key)["Body"].read()
image = Image.open(BytesIO(body)).convert("RGB")

plt.figure(figsize=(8, 6))
plt.imshow(image)
plt.title(image_key)
plt.axis("off")
plt.show()


## What does the collection distribution look like?

The collection is long-tailed: potato is the largest class, while soybean and cassava are the smallest. The chart below uses the verified counts from the cleaned dataset report.


In [ ]:
crop_counts = {
    "Potato": 1689917,
    "Sorghum": 730682,
    "Pigeon pea": 698011,
    "Finger millet": 593461,
    "Cowpea": 519759,
    "Sweet potato": 462894,
    "Groundnut": 411948,
    "Banana": 334732,
    "Pearl millet": 211390,
    "Yam": 117494,
    "Lentil": 78218,
    "Chickpea": 73856,
    "Rice": 53170,
    "Wheat": 37023,
    "Maize": 23348,
    "Common bean": 22295,
    "Cassava": 12120,
    "Soybean": 11237,
}

names = list(reversed(crop_counts.keys()))
counts = [crop_counts[name] for name in names]

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(names, counts, color="#2b6f4e")
ax.set_xlabel("Number of images")
ax.set_ylabel("Crop")
ax.set_title("ImageSafari cleaned images by crop")
ax.spines[["top", "right"]].set_visible(False)
ax.ticklabel_format(axis="x", style="plain")
plt.tight_layout()
plt.show()

print(f"Total images: {sum(crop_counts.values()):,}")


## One answered question

**How imbalanced is the crop distribution?** Potato contributes 1,689,917 images, or approximately 27.8% of the collection. The six largest classes account for most images, while several crops contribute fewer than 100,000 images. Accuracy alone may therefore conceal poor performance on smaller classes. Macro-averaged metrics and per-crop reporting are recommended.


In [ ]:
total = sum(crop_counts.values())
largest_crop = max(crop_counts, key=crop_counts.get)
largest_share = 100 * crop_counts[largest_crop] / total

print(f"Largest crop: {largest_crop}")
print(f"Images: {crop_counts[largest_crop]:,}")
print(f"Share: {largest_share:.1f}%")


## One open research question

**How well do crop-recognition models generalize to countries, centres, seasons, and capture methods that were not represented during training?** A useful study would build group-aware splits that hold out entire centres or countries, compare pretrained visual encoders, report macro F1 and per-crop recall, and measure performance separately for each held-out domain. Near-duplicate screening should be applied before splitting to reduce leakage.


## License and citation

The dataset is released under the [Creative Commons Attribution-ShareAlike 4.0 International License](https://creativecommons.org/licenses/by-sa/4.0/). Adapted material must be distributed under the same license.

Suggested citation: Alliance of Bioversity International and CIAT (2026). *ImageSafari Crop Image Collection*. Registry of Open Data on AWS.
